# Falcon 9 Launch Records: Web Scraping

Repair of the submitted web-scraping lab. The original file is unchanged.

Source: Wikipedia historical revision 1027686922 (the revision used by the original lab). This notebook removes citation markers, preserves full cell text, and rebuilds the records from a fresh parse. It does not replace the separate course datasets used for SQL, EDA, or machine learning.

Scope: one record per numbered Falcon 9 launch, using the primary payload row. Secondary payload rows and Falcon Heavy records are excluded. Payload mass text is retained as published; missing mass is not converted to zero.

The saved results were generated locally from the retrieved historical HTML. Network retrieval may depend on the environment.


## Setup
If needed, install `requests`, `beautifulsoup4`, and `pandas` in your Python environment.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

## Task 1: Retrieve the historical webpage

In [2]:
static_url = 'https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922'
response = requests.get(static_url, headers={'User-Agent': 'Mozilla/5.0 (educational data analysis)'}, timeout=60)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title.get_text(strip=True))

List of Falcon 9 and Falcon Heavy launches - Wikipedia


## Task 2: Inspect table columns without changing the source
Remove citation elements from a copy, not from the source tree. Read the complete cell rather than only its first hyperlink.

In [3]:
import re
from bs4 import BeautifulSoup
import pandas as pd

def clean_cell(cell):
    # Work on a copy so that inspecting headers cannot change the source table.
    copy = BeautifulSoup(str(cell), 'html.parser')
    for tag in copy.select('sup.reference, sup.mw-ref, style, script, .mw-editsection'):
        tag.decompose()
    text = copy.get_text(' ', strip=True).replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\s+([,;)])', r'\1', text)
    return re.sub(r'([([])\s+', r'\1', text)

def parse_launch_tables(soup):
    records = []
    # One record per numbered Falcon 9 launch. For launches with multiple
    # payload rows, retain the primary payload row, as in the course exercise.
    for table in soup.select('table.wikitable'):
        headings = ' '.join(clean_cell(h) for h in table.select('tr:first-child th'))
        if not all(word in headings for word in ('Flight', 'Payload', 'Booster')):
            continue
        for row in table.find_all('tr'):
            heading = row.find('th', recursive=False)
            if heading is None:
                continue
            flight = clean_cell(heading)
            if not re.fullmatch(r'\d+', flight):
                continue
            cells = row.find_all('td', recursive=False)
            if len(cells) != 9 or any(int(c.get('colspan', 1)) != 1 for c in cells):
                raise ValueError(f'Unexpected table structure at flight {flight}')
            values = [clean_cell(c) for c in cells]
            match = re.fullmatch(r'(.+?),?\s+(\d{1,2}:\d{2}(?::\d{2})?)(?:\s+UTC)?', values[0])
            if not match:
                raise ValueError(f'Unexpected date/time at flight {flight}: {values[0]}')
            records.append({
                'Flight No.': int(flight),
                'Launch site': values[2],
                'Payload': values[3],
                # Blank mass stays missing; it does not mean zero kilograms.
                'Payload mass': values[4] or None,
                'Orbit': values[5],
                'Customer': values[6],
                'Launch outcome': values[7],
                'Version Booster': values[1],
                'Booster landing': values[8],
                'Date': match.group(1).rstrip(','),
                'Time': match.group(2),
            })
    result = pd.DataFrame(records)
    if result.empty:
        raise ValueError('No numbered launch records found')
    if result['Flight No.'].duplicated().any():
        raise ValueError('Duplicate flight numbers found')
    return result.sort_values('Flight No.').reset_index(drop=True)


In [4]:
launch_tables = [t for t in soup.select('table.wikitable') if all(w in ' '.join(clean_cell(h) for h in t.select('tr:first-child th')) for w in ('Flight', 'Payload', 'Booster'))]
column_names = [clean_cell(h) for h in launch_tables[0].select('tr:first-child th')]
print(column_names)

['Flight No.', 'Date and time (UTC)', 'Version, Booster', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']


## Task 3: Parse launch records into a DataFrame

In [5]:
df = parse_launch_tables(soup)
df.head()

Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
1,"CCAFS, SLC-40",Dragon Spacecraft Qualification Unit,None,LEO,SpaceX,Success,F9 v1.0 B0003.1,Failure (parachute),4 June 2010,18:45
2,"CCAFS, SLC-40",Dragon demo flight C1 (Dragon C101),None,LEO (ISS),NASA (COTS) NRO,Success,F9 v1.0 B0004.1,Failure (parachute),8 December 2010,15:43
3,"CCAFS, SLC-40",Dragon demo flight C2+ (Dragon C102),"525 kg (1,157 lb)",LEO (ISS),NASA (COTS),Success,F9 v1.0 B0005.1,No attempt,22 May 2012,07:44
4,"CCAFS, SLC-40",SpaceX CRS-1 (Dragon C103),"4,700 kg (10,400 lb)",LEO (ISS),NASA (CRS),Success,F9 v1.0 B0006.1,No attempt,8 October 2012,00:35
5,"CCAFS, SLC-40",SpaceX CRS-2 (Dragon C104),"4,877 kg (10,752 lb)",LEO (ISS),NASA (CRS),Success,F9 v1.0 B0007.1,No attempt,1 March 2013,15:10


## Validate representative records
Checks include complete site and customer text, distinct early payloads, citation-free booster identifiers, and missing-mass handling.

In [6]:
# Check representative records against the historical source table.
assert df['Flight No.'].is_unique
assert df.iloc[0]['Version Booster'] == 'F9 v1.0 B0003.1'
assert 'SLC-40' in df.iloc[0]['Launch site']
assert 'demo flight C1' in df.iloc[1]['Payload']
assert 'demo flight C2+' in df.iloc[2]['Payload']
assert 'COTS' in df.iloc[1]['Customer'] and 'NRO' in df.iloc[1]['Customer']
assert df.iloc[0]['Booster landing'] == 'Failure (parachute)'
assert pd.isna(df.iloc[0]['Payload mass'])
assert not df['Version Booster'].str.contains(r'\[\d+\]', regex=True).any()
print(f'Validation passed for {len(df)} numbered Falcon 9 launch records.')


Validation passed for 121 numbered Falcon 9 launch records.


In [7]:
print('Records:', len(df))
print('Missing payload mass:', df['Payload mass'].isna().sum())
print('Date range:', pd.to_datetime(df['Date']).min().date(), 'to', pd.to_datetime(df['Date']).max().date())

Records: 121
Missing payload mass: 2
Date range: 2010-06-04 to 2021-06-06


## Export
This export is web-scraped evidence. Do not substitute it for the different preselected course datasets without reconciling dates, scope, units, and field definitions.

In [8]:
df.to_csv('spacex_web_scraped.csv', index=False)
print('Saved spacex_web_scraped.csv')

Saved spacex_web_scraped.csv
